In [ ]:
import sys
from pathlib import Path

print("Current dir:", Path.cwd())
sys.path.append(str(Path.cwd().parent))

import config as config
print("Loaded from:", config.__file__)

Setting up session and reading data from cloud(s3)

In [ ]:
from config import get_spark_session, s3_path, BUCKET_NAME
from pyspark.sql.functions import*
spark = get_spark_session("bronze-to-silver")

products_df = spark.read.csv(
    s3_path("bronze", "products", "olist_products_dataset.csv"),
    header=True,
    inferSchema=True
)

products_df.show(5)

Data understanding, Before coding the transformation we should understand the each aspect of the data.

In [ ]:
print(f"columns: {products_df.columns}")
print(f"Number of records: {products_df.count()}")
print(products_df.printSchema())


Checks Nulls in each row

In [ ]:
products_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in products_df.columns
]).show()

Primary key quality check

In [ ]:
products_df.groupBy("product_id").count().filter("count > 1").show()

everything once 

In [ ]:
products_df.printSchema()

products_df.count()

products_df.show(5, truncate=False)

products_df.describe().show()

from pyspark.sql.functions import col, count, when

products_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in products_df.columns
]).show()

products_df.groupBy("product_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

Start transforming table after investigation 


In [ ]:
products_df = (
    products_df
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed(
        "product_description_lenght",
        "product_description_length"
    )
)

from pyspark.sql.functions import trim, col

products_df = (
    products_df
    .withColumn("product_id", trim(col("product_id")))
    .withColumn(
        "product_category_name",
        lower(trim(col("product_category_name")))
    )
)



In [ ]:
products_df.select("product_category_name") \
    .distinct() \
    .orderBy("product_category_name") \
    .show(100, truncate=False)

In [ ]:
products_df.filter(
    col("product_category_name").isNull()
).show(20, truncate=False)

In [ ]:
products_df.filter(
    col("product_weight_g").isNull()
).show(truncate=False)